# LoRA fine-tune: Qwen2.5-3B — классификатор тикетов поддержки jWorkPlace

Учебный ноутбук (сайдкар `experiments/finetune/`, не прод-код backend'а jWorkPlace). Тюним
`Qwen2.5-3B-Instruct` через unsloth (4bit + LoRA) на классификацию тикетов поддержки по
фиксированной таксономии 6 категорий x 3 приоритета (см. `taxonomy.md`), затем экспортируем
merged GGUF для запуска в Ollama на нашем VPS.

## Как загрузить это в Kaggle

1. Создайте новый Kaggle Notebook, включите **GPU T4** (Settings → Accelerator → GPU T4 x2 или x1).
2. Загрузите `data/train.jsonl` и `data/eval.jsonl` как Kaggle Dataset (**Add Data → Upload**), либо
   через `Add Data → Upload` прямо в сессию ноутбука (кнопка справа).
3. Обновите переменные путей в ячейке ниже (`TRAIN_PATH`, `EVAL_PATH`) под то, куда Kaggle примонтировал
   файлы (обычно `/kaggle/input/<dataset-name>/train.jsonl`).
4. **Run All.** Ноутбук самодостаточен: baseline снимается ДО тюна, тюн — на 40 train-примерах,
   AFTER-метрика — на тех же 10 eval, что и baseline (честное сравнение).
5. В конце — ячейка экспорта merged GGUF q4_k_m и инструкция подключить в Ollama на VPS.


In [ ]:
# (a) Установка unsloth + зависимостей (GPU T4, CUDA уже в Kaggle-образе).
# ВАЖНО: базовый образ Kaggle приезжает с несовместимыми версиями
# (transformers слишком новый, trl слишком старый) — пиним строго под окно unsloth.
!pip install -q -U unsloth unsloth_zoo
!pip install -q "transformers>=4.51.3,<=5.5.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" accelerate peft bitsandbytes datasets

## (b) Базовая модель: Qwen2.5-3B-Instruct в 4bit + LoRA-адаптер

`r=16` — стандартный компромисс качество/скорость для маленького датасета (40 примеров) на T4.
`target_modules` — типовой набор проекций attention+MLP для Qwen2.5 (unsloth автоопределяет имена
модулей, но перечисляем явно для воспроизводимости).

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024  # с запасом: system+user+короткий JSON-ответ укладываются с большим запасом
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,       # авто (bfloat16 на T4, если поддерживается)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


## (c) Загрузка датасета и форматирование под Qwen chat template

Пути ниже — под Kaggle Dataset upload; поправьте `TRAIN_PATH`/`EVAL_PATH`, если Kaggle примонтировал
файлы в другую директорию (см. панель Data справа).

In [ ]:
import json
from datasets import Dataset

TRAIN_PATH = "/kaggle/input/jworkplace-ticket-classification/train.jsonl"
EVAL_PATH = "/kaggle/input/jworkplace-ticket-classification/eval.jsonl"


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


train_records = load_jsonl(TRAIN_PATH)
eval_records = load_jsonl(EVAL_PATH)
print(f"train: {len(train_records)} примеров, eval: {len(eval_records)} примеров")


def to_chat_text(record):
    # apply_chat_template с add_generation_prompt=False — тренируем на полном диалоге,
    # включая assistant-ответ (нужен для SFT на chat-формате).
    return tokenizer.apply_chat_template(
        record["messages"], tokenize=False, add_generation_prompt=False
    )


train_texts = [{"text": to_chat_text(r)} for r in train_records]
train_dataset = Dataset.from_list(train_texts)
print(train_dataset[0]["text"][:500])


## (d) BEFORE-baseline: нетюненая модель на 10 eval-примерах

Это baseline для итогового сравнения (та же метрика, что считает `baseline_runner.py` локально:
per-field accuracy category/priority, valid_json_ratio, exact-match).

In [ ]:
import re


def parse_model_json(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[len("json"):]
        text = text.strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def generate_reply(model, tokenizer, system, user, max_new_tokens=40):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    FastLanguageModel.for_inference(model)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def eval_model(model, tokenizer, eval_records, label=""):
    results = []
    cat_correct = prio_correct = exact_correct = valid_json = 0
    for rec in eval_records:
        system = rec["messages"][0]["content"]
        user = rec["messages"][1]["content"]
        gold = json.loads(rec["messages"][2]["content"])

        raw = generate_reply(model, tokenizer, system, user)
        parsed = parse_model_json(raw)

        cat_ok = bool(parsed) and parsed.get("category") == gold["category"]
        prio_ok = bool(parsed) and parsed.get("priority") == gold["priority"]
        if parsed is not None:
            valid_json += 1
        if cat_ok:
            cat_correct += 1
        if prio_ok:
            prio_correct += 1
        if cat_ok and prio_ok:
            exact_correct += 1

        results.append({"user": user, "gold": gold, "raw": raw, "parsed": parsed})

    n = len(eval_records)
    summary = {
        "label": label,
        "n": n,
        "valid_json_ratio": valid_json / n,
        "category_accuracy": cat_correct / n,
        "priority_accuracy": prio_correct / n,
        "exact_match_accuracy": exact_correct / n,
        "results": results,
    }
    return summary


baseline_summary = eval_model(model, tokenizer, eval_records, label="baseline (до LoRA)")
print(json.dumps({k: v for k, v in baseline_summary.items() if k != "results"}, ensure_ascii=False, indent=2))

with open("baseline_before_lora.json", "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, ensure_ascii=False, indent=2)


## (e) Тренировка SFTTrainer

Маленький датасет (40 примеров) → 3 эпохи с малым batch size и gradient accumulation, чтобы
уложиться в память T4. `packing=False` — примеры короткие, не увеличиваем сложность отладки упаковкой.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()


## (f) AFTER: тюненая модель на тех же 10 eval-примерах

In [ ]:
tuned_summary = eval_model(model, tokenizer, eval_records, label="tuned (после LoRA)")
print(json.dumps({k: v for k, v in tuned_summary.items() if k != "results"}, ensure_ascii=False, indent=2))

with open("tuned_after_lora.json", "w", encoding="utf-8") as f:
    json.dump(tuned_summary, f, ensure_ascii=False, indent=2)


## (g) Таблица baseline vs tuned (per-field accuracy)

Критерии оценки — см. `taxonomy.md` (раздел «Критерии оценки «стало лучше»»): exact-match accuracy
и valid_json_ratio — главные критерии успеха тюна; per-field accuracy — диагностика.

In [ ]:
def print_comparison(baseline, tuned):
    metrics = [
        ("valid_json_ratio", "Доля валидного JSON"),
        ("category_accuracy", "Accuracy category"),
        ("priority_accuracy", "Accuracy priority"),
        ("exact_match_accuracy", "Exact-match (оба поля)"),
    ]
    header = f"{'Метрика':<28} {'baseline':>10} {'tuned':>10} {'delta':>10}"
    print(header)
    print("-" * len(header))
    for key, title in metrics:
        b, t = baseline[key], tuned[key]
        print(f"{title:<28} {b:>9.1%} {t:>9.1%} {t - b:>+9.1%}")


print_comparison(baseline_summary, tuned_summary)


## (h) Экспорт merged GGUF (q4_k_m) + подключение в Ollama на VPS

Экспортируем merged (base+LoRA) веса сразу в GGUF q4_k_m — квантование, совместимое с Ollama
и укладывающееся в RAM VPS (3.8 ГБ, см. `../../CLAUDE.md`).

In [ ]:
model.save_pretrained_gguf(
    "jwp-ticket-classifier-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
# Файл появится как jwp-ticket-classifier-gguf/*.gguf в файлах сессии Kaggle — скачайте его
# через панель Output справа (Kaggle сохраняет вывод ноутбука как Kaggle Output после Save & Run).


### Как подключить GGUF в Ollama на нашем VPS

1. Скачайте `.gguf`-файл из Kaggle Output (Save Version → Output файлы) на VPS, например в
   `~/models/jwp-ticket-classifier.gguf`.
2. Создайте `Modelfile` рядом:
   ```
   FROM ~/models/jwp-ticket-classifier.gguf
   ```
3. Соберите модель в Ollama:
   ```bash
   ollama create jwp-ticket-classifier -f Modelfile
   ```
4. Прогоните `baseline_runner.py` уже против неё локально на VPS (та же метрика, что и в этом
   ноутбуке — честное сравнение с cloud-baseline):
   ```bash
   python3 experiments/finetune/baseline_runner.py \
       --base-url http://localhost:11434/v1 --model jwp-ticket-classifier
   ```
   ⚠️ Помните про `MAX_LOADED_MODELS=1` и лимит RAM 3.8 ГБ на VPS (см. память проекта) — не
   держите одновременно загруженными `nomic-embed` и тюненую модель классификатора без нужды.
